### 1. Environment & Setup
This section handles the initial setup of the computational environment. It installs the `unsloth` library for highly optimized, memory-efficient LLM inference, and mounts Google Drive so the notebook can access the necessary datasets and pre-trained adapter models.

In [ ]:
# Unsloth "Nuclear" installation script
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-bcnb0ehx/unsloth_acc3bd0d347d4871a0dc7653fbb215fc
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-bcnb0ehx/unsloth_acc3bd0d347d4871a0dc7653fbb215fc
  Resolved https://github.com/unslothai/unsloth.git to commit 2ef394137aef657f9578aab57813361366e72c17
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 152.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 117.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 23.5 MB/s eta 0:00:00
  

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import re

# Base Paths
BASE_PATH = "/content/drive/MyDrive/colab_data/HIPE-2026-data"
AT_ADAPTER = os.path.join(BASE_PATH, "trained_models/llama_8b_at_adapter")
ISAT_ADAPTER = os.path.join(BASE_PATH, "trained_models/llama_8b_isAt_adapter")
DATA_PATH = os.path.join(BASE_PATH, "data/sandbox")

LANGUAGES = ['en', 'de', 'fr']

### 2. Specialized Logic Requirements
Here we define essential utility functions for the pipeline. This includes data loading routines to parse the `.jsonl` files and a robust JSON harvester (`harvest_json_robust`) designed to safely extract the model's predictions even if the output contains minor formatting errors or hallucinations.

In [ ]:
def harvest_json_robust(text):
    # Normalize smart quotes
    text = text.replace('“', '"').replace('”', '"').replace('‘', "'").replace('’', "'")

    try:
        # Attempt to parse the entire text as a single JSON object
        full_json = json.loads(text)
        # If it's a dictionary with a 'results' key that's a non-empty list
        if isinstance(full_json, dict) and "results" in full_json and \
           isinstance(full_json["results"], list) and len(full_json["results"]) > 0:
            # Return the first item from the 'results' list, wrapped in a list
            # This is to make it compatible with the existing iteration logic `for p in parsed:`
            return [full_json["results"][0]]
        # If it's a dictionary but doesn't have the 'results' structure, return it as is (wrapped in a list)
        elif isinstance(full_json, dict):
            return [full_json]
    except json.JSONDecodeError:
        # If direct parsing fails, proceed to regex-based extraction
        pass

    # Fallback: Extract JSON objects using regex (original logic, for simple cases or partial outputs)
    matches = re.findall(r'\{[^{}]*\}', text)
    results = []
    for match in matches:
        # Correct unquoted labels
        match = re.sub(r':\s*TRUE\b', ': "TRUE"', match, flags=re.IGNORECASE)
        match = re.sub(r':\s*FALSE\b', ': "FALSE"', match, flags=re.IGNORECASE)
        match = re.sub(r':\s*PROBABLE\b', ': "PROBABLE"', match, flags=re.IGNORECASE)
        try:
            results.append(json.loads(match))
        except json.JSONDecodeError:
            pass

    # Return results if any were found, otherwise an empty list to indicate no valid JSON was harvested
    return results if results else []

def load_data(lang):
    filepath = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")
    data = []
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line))
    else:
        print(f"Warning: {filepath} not found.")
    return data

### 3. Prompt Templates
This section outlines our prompt engineering strategy. We define strict, structured templates for both the `at` (structural/institutional connection) and `isAt` (immediate physical presence) relation extraction tasks. These prompts enforce the logical constraints and guarantee that the model outputs parsable JSON.

In [ ]:
at_prompt = """You are an expert computational historian specializing in relation extraction.
TASK: Determine the historical relation 'at' between the designated Persons and Places based on the text.

CRITICAL LOGICAL CONSTRAINTS:
1. 'at' represents a permanent, structural, institutional, professional, or residency-based geographic connection over time.
2. Use 'TRUE' ONLY if there is 100% certainty and explicit absolute proof of the connection.
3. Use 'PROBABLE' for 'at' when strong contextual, regional, or family/professional affiliation implies geographic connectivity without explicit absolute proof.
4. Use 'FALSE' if no evidence is present or the context contradicts such a relation.

TARGET TEXT FOR ANALYSIS:
"{text}"

### OUTPUT INSTRUCTIONS ###
- Evaluate the relationships for the requested pairs.
- Output ONLY the JSON object. START your response with '{{'.
- DO NOT output a "reasoning" field, explanations, or any preamble text.
- The JSON must follow this exact format: {{"at": "VALUE"}} where VALUE is TRUE, FALSE, or PROBABLE.

PAIRS TO EVALUATE:
{pairs_list_str}"""

isAt_prompt = """You are an expert computational historian specializing in relation extraction.
TASK: Determine the temporal relation 'isAt' between the designated Persons and Places based on the text.

CRITICAL LOGICAL CONSTRAINTS:
1. 'isAt' represents literal immediate physical presence at that place within the narrative moment (the temporal horizon of the article).
2. 'isAt' is TRUE if there is evidence the person was at the location up to about one month before the publication date.
3. Use 'FALSE' if the person is elsewhere, the event happened in the distant past, or no evidence of current presence exists.

TARGET TEXT FOR ANALYSIS:
"{text}"

### OUTPUT INSTRUCTIONS ###
- Evaluate the relationships for the requested pairs.
- Output ONLY the JSON object. START your response with '{{'.
- DO NOT output a "reasoning" field, explanations, or any preamble text.
- The JSON must follow this exact format: {{"isAt": "VALUE"}} where VALUE is TRUE or FALSE.

PAIRS TO EVALUATE:
{pairs_list_str}"""

def format_chat_prompt(relation, person, place, text):
    pairs_list_str = f"Person: {person}, Place: {place}"
    if relation == 'at':
        user_msg = at_prompt.format(text=text, pairs_list_str=pairs_list_str)
    else:
        user_msg = isAt_prompt.format(text=text, pairs_list_str=pairs_list_str)

    return [{"role": "user", "content": user_msg}]

### 4. Sequential Inference
This is the core execution block. We employ a sequential Parameter-Efficient Fine-Tuning (PEFT) strategy to maximize VRAM efficiency. First, we load the base model with the `at` adapter and process the documents. Then, we flush the memory and switch to the `isAt` adapter to evaluate the temporal relations, appending all predictions to the dataset.

In [ ]:
from unsloth import FastLanguageModel
import torch
import sys

max_seq_length = 4096

if not os.path.exists(AT_ADAPTER):
    print(f"Error: AT adapter path does not exist: {AT_ADAPTER}")
    sys.exit(1)

# Load base model
print("Loading Base Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

Loading Base Model...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


In [ ]:
from tqdm.notebook import tqdm

# 1. AT Inference
print(f"Loading AT-Specialist from {AT_ADAPTER}...")
if "at_adapter" not in getattr(model, "peft_config", {}):
    model.load_adapter(AT_ADAPTER, adapter_name="at_adapter")
model.set_adapter("at_adapter")
FastLanguageModel.for_inference(model)

run_lang = "fr"

if 'at_predictions' not in locals():
    at_predictions = {}
at_predictions[run_lang] = []

print(f"Running AT inference for {run_lang.upper()}...")
data = load_data(run_lang)

for item in tqdm(data, desc="Processing Documents (AT)"):
    for pair in item.get('sampled_pairs', []):
        pers_list = pair.get('pers_mentions_list', [])
        loc_list = pair.get('loc_mentions_list', [])
        person = pers_list[0] if pers_list else ""
        place = loc_list[0] if loc_list else ""

        messages = format_chat_prompt('at', person, place, item.get('text', ''))
        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

        max_retries = 3
        for attempt in range(max_retries):
            # Use short tokens and stop strings to prevent reasoning bloat
            outputs = model.generate(input_ids=inputs, max_new_tokens=20, do_sample=False, stop_strings=["}"], tokenizer=tokenizer)
            response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

            resp_clean = response.strip().upper()
            if resp_clean in ['TRUE', 'FALSE', 'PROBABLE']:
                pred = resp_clean
            else:
                parsed = harvest_json_robust(response)
                pred = 'ERROR'
                for p in parsed:
                    if isinstance(p, dict):
                        for val in p.values():
                            if str(val).upper() in ['TRUE', 'FALSE', 'PROBABLE']:
                                pred = str(val).upper()
                                break
                    if pred != 'ERROR':
                        break

                if pred == 'ERROR':
                    if 'PROBABLE' in resp_clean:
                        pred = 'PROBABLE'
                    elif 'TRUE' in resp_clean:
                        pred = 'TRUE'
                    elif 'FALSE' in resp_clean:
                        pred = 'FALSE'

            if pred != 'ERROR':
                break
            elif attempt < max_retries - 1:
                print(f"  [AT Retry {attempt+1}] Model returned error, re-prompting...")

        if pred == 'ERROR':
            print(f"[AT ERROR] doc: {item.get('document_id')} | pair: {person}-{place} | Raw Response: {response}")
            pred = 'FALSE' # Safety net to prevent schema failure

        pair['at'] = pred

    at_predictions[run_lang].append(item)

Loading AT-Specialist from /content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_8b_at_adapter...
Running AT inference for FR...


Processing Documents (AT):   0%|          | 0/107 [00:00<?, ?it/s]

Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

In [ ]:
import torch
import gc

# Delete the model and trainer from memory
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# Now load the base model fresh
print("Loading Base Model for ISAT...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)

# Load the isAt adapter
print(f"Loading ISAT-Specialist from {ISAT_ADAPTER}...")
model.load_adapter(ISAT_ADAPTER, adapter_name="isAt_adapter")
model.set_adapter("isAt_adapter")
FastLanguageModel.for_inference(model)

print("Ready to run ISAT inference loop.")


Loading Base Model for ISAT...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


Loading ISAT-Specialist from /content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_8b_isAt_adapter...


Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

Ready to run ISAT inference loop.


In [ ]:
from tqdm.notebook import tqdm
import time
import json
import os

# 2. ISAT Inference
run_lang = "fr"

if 'final_results' not in locals():
    final_results = {}
final_results[run_lang] = []

print(f"Running ISAT inference for {run_lang.upper()}...")

if run_lang not in at_predictions or not at_predictions[run_lang]:
    print(f"Error: You must run the AT Inference block for '{run_lang}' first!")
else:
    total_docs = len(at_predictions[run_lang])
    for doc_idx, item in enumerate(at_predictions[run_lang]):
        doc_id = item.get('document_id', 'Unknown')
        pairs = item.get('sampled_pairs', [])


        for i, pair in enumerate(pairs):
            pers_list = pair.get('pers_mentions_list', [])
            loc_list = pair.get('loc_mentions_list', [])
            person = pers_list[0] if pers_list else ""
            place = loc_list[0] if loc_list else ""

            messages = format_chat_prompt('isAt', person, place, item.get('text', ''))
            inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

            max_retries = 3
            for attempt in range(max_retries):
                start_time = time.time()
                # Use stop_strings and very low max_new_tokens to force the model to stay on topic
                outputs = model.generate(input_ids=inputs, max_new_tokens=15, do_sample=False, use_cache=True, stop_strings=["}"], tokenizer=tokenizer)
                end_time = time.time()

                gen_duration = end_time - start_time
                response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

                resp_clean = response.strip().upper()
                if resp_clean in ['TRUE', 'FALSE']:
                    isAt_pred = resp_clean
                else:
                    parsed = harvest_json_robust(response)
                    isAt_pred = 'ERROR'
                    for p in parsed:
                        if isinstance(p, dict):
                            for val in p.values():
                                if str(val).upper() in ['TRUE', 'FALSE']:
                                    isAt_pred = str(val).upper()
                                    break
                        if isAt_pred != 'ERROR':
                            break

                if isAt_pred == 'ERROR':
                    if 'TRUE' in resp_clean:
                        isAt_pred = 'TRUE'
                    elif 'FALSE' in resp_clean:
                        isAt_pred = 'FALSE'

                if isAt_pred != 'ERROR':
                    break
                elif attempt < max_retries - 1:
                    print(f"    [ISAT Retry {attempt+1}] Model returned error, re-prompting...")

            if isAt_pred == 'ERROR':
                print(f"[isAt ERROR] doc: {doc_id} | pair: {person}-{place} | Raw Response: {response}")
                isAt_pred = 'FALSE' # Safety net to prevent schema failure

            pair['isAt'] = isAt_pred

        final_results[run_lang].append(item)

    out_path = os.path.join(BASE_PATH, f"integrated_llama_8b_{run_lang}_results.jsonl")
    with open(out_path, 'w', encoding='utf-8') as f:
        for res in final_results[run_lang]:
            f.write(json.dumps(res) + '\n')
    print(f"\nDONE. Saved final predictions to {out_path}")

Both `max_new_tokens` (=15) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running ISAT inference for FR...


Both `max_new_tokens` (=15) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=15) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=15) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=15) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

    [ISAT Retry 1] Model returned error, re-prompting...


Both `max_new_tokens` (=15) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

    [ISAT Retry 2] Model returned error, re-prompting...


Both `max_new_tokens` (=15) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[isAt ERROR] doc: GDL-1921-07-17-15 | pair: M. Tarabori-Bellinzone | Raw Response: {"reasoning": "Extracting based on historical context. Evaluating is


Both `max_new_tokens` (=15) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=15) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=15) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=15) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


DONE. Saved final predictions to /content/drive/MyDrive/colab_data/HIPE-2026-data/integrated_llama_8b_fr_results.jsonl


### 5. Evaluation
Once inference is complete, this section assesses the model's performance. It executes official scoring scripts against the gold standard annotations and provides a detailed breakdown of accuracy, recall, and label distributions across the tested languages (English, German, French).

In [ ]:
import json
import os

print("Running automated evaluation script...")

for lang in LANGUAGES:
    pred_path = os.path.join(BASE_PATH, f"integrated_llama_8b_{lang}_results.jsonl")
    gold_path = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")

    if os.path.exists(pred_path) and os.path.exists(gold_path):
        print(f"\nEvaluating {lang}...")
        # cd into BASE_PATH so the script can find the 'schemas/' directory
        !cd "{BASE_PATH}" && python scripts/file_scorer_evaluation.py --predictions_file "{pred_path}" --gold_data_file "{gold_path}"
    else:
        print(f"Skipping eval for {lang}. Check if prediction or gold files exist.")

Running automated evaluation script...

Evaluating en...

Evaluation Results for integrated_llama_8b_en_results.jsonl:
  'at': macro_recall=0.6105, accuracy=0.5497 (83/151)
  'isAt': macro_recall=0.8204, accuracy=0.7682 (116/151)
  'global': macro_recall=0.7154 (199/302)


Evaluating de...

Evaluation Results for integrated_llama_8b_de_results.jsonl:
  'at': macro_recall=0.5764, accuracy=0.5718 (247/432)
  'isAt': macro_recall=0.7533, accuracy=0.8681 (375/432)
  'global': macro_recall=0.6648 (622/864)


Evaluating fr...

Evaluation Results for integrated_llama_8b_fr_results.jsonl:
  'at': macro_recall=0.5320, accuracy=0.5547 (831/1498)
  'isAt': macro_recall=0.7459, accuracy=0.8618 (1291/1498)
  'global': macro_recall=0.6389 (2122/2996)



In [ ]:
import json
import os

print("Detailed Prediction Analysis by Label")

for lang in LANGUAGES:
    pred_path = os.path.join(BASE_PATH, f"integrated_llama_8b_{lang}_results.jsonl")
    gold_path = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")

    if not (os.path.exists(pred_path) and os.path.exists(gold_path)):
        print(f"Missing files for {lang}")
        continue

    print(f"\n{'='*40}")
    print(f"--- Analysis for {lang.upper()} ---")
    print(f"{'='*40}")

    # 1. Load gold data into a dictionary mapped by document_id
    gold_data = {}
    with open(gold_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            doc_id = item['document_id']
            # Map pair entities to their gold pairs so order doesn't matter
            gold_data[doc_id] = {}
            for pair in item.get('sampled_pairs', []):
                pers_id = pair.get('pers_entity_id')
                loc_id = pair.get('loc_entity_id')
                gold_data[doc_id][(pers_id, loc_id)] = pair

    # 2. Track stats
    at_stats = {
        "TRUE": {"correct": 0, "wrong": 0},
        "FALSE": {"correct": 0, "wrong": 0},
        "PROBABLE": {"correct": 0, "wrong": 0}
    }
    isat_stats = {
        "TRUE": {"correct": 0, "wrong": 0},
        "FALSE": {"correct": 0, "wrong": 0}
    }

    # 3. Compare predictions against gold
    with open(pred_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            doc_id = item['document_id']
            pred_pairs = item.get('sampled_pairs', [])

            for p_pair in pred_pairs:
                pers_id = p_pair.get('pers_entity_id')
                loc_id = p_pair.get('loc_entity_id')

                # Find corresponding gold pair
                g_pair = gold_data.get(doc_id, {}).get((pers_id, loc_id))

                if not g_pair:
                    continue # Skip if no matching gold pair found

                # --- Check 'at' Field ---
                g_at = g_pair.get('at', 'FALSE')
                p_at = p_pair.get('at', 'ERROR')

                if g_at in at_stats:
                    if g_at == p_at:
                        at_stats[g_at]['correct'] += 1
                    else:
                        at_stats[g_at]['wrong'] += 1

                # --- Check 'isAt' Field ---
                g_isat = g_pair.get('isAt', 'FALSE')
                p_isat = p_pair.get('isAt', 'ERROR')

                if g_isat in isat_stats:
                    if g_isat == p_isat:
                        isat_stats[g_isat]['correct'] += 1
                    else:
                        isat_stats[g_isat]['wrong'] += 1

    # 4. Print Results
    print("\n[ 'AT' FIELD STATS ]")
    for label, counts in at_stats.items():
        total = counts['correct'] + counts['wrong']
        acc = (counts['correct'] / total * 100) if total > 0 else 0
        print(f"  Gold={label:<8}: {counts['correct']:>4} Correct | {counts['wrong']:>4} Wrong | Accuracy: {acc:>5.1f}%")

    print("\n[ 'ISAT' FIELD STATS ]")
    for label, counts in isat_stats.items():
        total = counts['correct'] + counts['wrong']
        acc = (counts['correct'] / total * 100) if total > 0 else 0
        print(f"  Gold={label:<8}: {counts['correct']:>4} Correct | {counts['wrong']:>4} Wrong | Accuracy: {acc:>5.1f}%")


Detailed Prediction Analysis by Label

--- Analysis for EN ---

[ 'AT' FIELD STATS ]
  Gold=TRUE    :   28 Correct |    1 Wrong | Accuracy:  96.6%
  Gold=FALSE   :   40 Correct |   28 Wrong | Accuracy:  58.8%
  Gold=PROBABLE:   15 Correct |   39 Wrong | Accuracy:  27.8%

[ 'ISAT' FIELD STATS ]
  Gold=TRUE    :   16 Correct |    2 Wrong | Accuracy:  88.9%
  Gold=FALSE   :  100 Correct |   33 Wrong | Accuracy:  75.2%

--- Analysis for DE ---

[ 'AT' FIELD STATS ]
  Gold=TRUE    :   30 Correct |   11 Wrong | Accuracy:  73.2%
  Gold=FALSE   :  177 Correct |   67 Wrong | Accuracy:  72.5%
  Gold=PROBABLE:   40 Correct |  107 Wrong | Accuracy:  27.2%

[ 'ISAT' FIELD STATS ]
  Gold=TRUE    :   18 Correct |   11 Wrong | Accuracy:  62.1%
  Gold=FALSE   :  357 Correct |   46 Wrong | Accuracy:  88.6%

--- Analysis for FR ---

[ 'AT' FIELD STATS ]
  Gold=TRUE    :  114 Correct |   65 Wrong | Accuracy:  63.7%
  Gold=FALSE   :  594 Correct |  358 Wrong | Accuracy:  62.4%
  Gold=PROBABLE:  123 Correct

In [ ]:
import json
import os

print("Prediction Distribution (Model Guesses)")

for lang in LANGUAGES:
    pred_path = os.path.join(BASE_PATH, f"integrated_llama_8b_{lang}_results.jsonl")

    if not os.path.exists(pred_path):
        print(f"Missing prediction file for {lang}")
        continue

    print(f"\n{'='*40}")
    print(f"--- Model Guesses for {lang.upper()} ---")
    print(f"{'='*40}")

    at_guesses = {"TRUE": 0, "FALSE": 0, "PROBABLE": 0, "ERROR": 0}
    isat_guesses = {"TRUE": 0, "FALSE": 0, "ERROR": 0}

    with open(pred_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            for pair in item.get('sampled_pairs', []):
                p_at = pair.get('at', 'ERROR')
                p_isat = pair.get('isAt', 'ERROR')

                if p_at in at_guesses:
                    at_guesses[p_at] += 1
                else:
                    at_guesses['ERROR'] += 1

                if p_isat in isat_guesses:
                    isat_guesses[p_isat] += 1
                else:
                    isat_guesses['ERROR'] += 1

    print("\n[ 'AT' FIELD GUESSES ]")
    for label, count in at_guesses.items():
        if count > 0 or label in ["TRUE", "FALSE", "PROBABLE"]:
            print(f"  Guessed {label:<8}: {count:>4} times")

    print("\n[ 'ISAT' FIELD GUESSES ]")
    for label, count in isat_guesses.items():
        if count > 0 or label in ["TRUE", "FALSE"]:
            print(f"  Guessed {label:<8}: {count:>4} times")

Prediction Distribution (Model Guesses)

--- Model Guesses for EN ---

[ 'AT' FIELD GUESSES ]
  Guessed TRUE    :   79 times
  Guessed FALSE   :   41 times
  Guessed PROBABLE:   31 times

[ 'ISAT' FIELD GUESSES ]
  Guessed TRUE    :   49 times
  Guessed FALSE   :  102 times

--- Model Guesses for DE ---

[ 'AT' FIELD GUESSES ]
  Guessed TRUE    :  136 times
  Guessed FALSE   :  205 times
  Guessed PROBABLE:   91 times

[ 'ISAT' FIELD GUESSES ]
  Guessed TRUE    :   64 times
  Guessed FALSE   :  368 times

--- Model Guesses for FR ---

[ 'AT' FIELD GUESSES ]
  Guessed TRUE    :  409 times
  Guessed FALSE   :  688 times
  Guessed PROBABLE:  401 times

[ 'ISAT' FIELD GUESSES ]
  Guessed TRUE    :  234 times
  Guessed FALSE   : 1264 times


In [ ]:
import json
import os

print("Gold Label Distribution (Actual Data)")

for lang in LANGUAGES:
    gold_path = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")

    if not os.path.exists(gold_path):
        print(f"Missing gold file for {lang}")
        continue

    print(f"\n{'='*40}")
    print(f"--- Gold Labels for {lang.upper()} ---")
    print(f"{'='*40}")

    at_counts = {"TRUE": 0, "FALSE": 0, "PROBABLE": 0, "ERROR": 0}
    isat_counts = {"TRUE": 0, "FALSE": 0, "ERROR": 0}

    with open(gold_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            for pair in item.get('sampled_pairs', []):
                g_at = pair.get('at', 'ERROR')
                g_isat = pair.get('isAt', 'ERROR')

                if g_at in at_counts:
                    at_counts[g_at] += 1
                else:
                    at_counts['ERROR'] += 1

                if g_isat in isat_counts:
                    isat_counts[g_isat] += 1
                else:
                    isat_counts['ERROR'] += 1

    print("\n[ 'AT' FIELD GOLD LABELS ]")
    for label, count in at_counts.items():
        if count > 0 or label in ["TRUE", "FALSE", "PROBABLE"]:
            print(f"  Actual {label:<8}: {count:>4} times")

    print("\n[ 'ISAT' FIELD GOLD LABELS ]")
    for label, count in isat_counts.items():
        if count > 0 or label in ["TRUE", "FALSE"]:
            print(f"  Actual {label:<8}: {count:>4} times")

Gold Label Distribution (Actual Data)

--- Gold Labels for EN ---

[ 'AT' FIELD GOLD LABELS ]
  Actual TRUE    :   29 times
  Actual FALSE   :   68 times
  Actual PROBABLE:   54 times

[ 'ISAT' FIELD GOLD LABELS ]
  Actual TRUE    :   18 times
  Actual FALSE   :  133 times

--- Gold Labels for DE ---

[ 'AT' FIELD GOLD LABELS ]
  Actual TRUE    :   41 times
  Actual FALSE   :  244 times
  Actual PROBABLE:  147 times

[ 'ISAT' FIELD GOLD LABELS ]
  Actual TRUE    :   29 times
  Actual FALSE   :  403 times

--- Gold Labels for FR ---

[ 'AT' FIELD GOLD LABELS ]
  Actual TRUE    :  179 times
  Actual FALSE   :  952 times
  Actual PROBABLE:  367 times

[ 'ISAT' FIELD GOLD LABELS ]
  Actual TRUE    :  127 times
  Actual FALSE   : 1371 times
